## 1. Imports

In [27]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


## 2. Load the multi-city dataset

In [28]:
DATA_PATH = "../data/processed/multi_city_recommendation_features.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nCities:")
print(df["city"].value_counts())


Dataset shape: (120, 27)

Cities:
city
Manali       20
Goa          20
Jaipur       20
Udaipur      20
Rishikesh    20
Shimla       20
Name: count, dtype: int64


## 3. Standardize the recommendation score

The project has used several names for recommendation scores across earlier notebooks.

We standardize everything to:

```text
recommendation_score
```


In [29]:
def minmax(series):
    series = pd.to_numeric(
        series,
        errors="coerce"
    ).fillna(0.0)

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            np.ones(len(series)),
            index=series.index
        )

    return (
        (series - minimum)
        / (maximum - minimum)
    )


score_column = None

for candidate in [
    "final_score",
    "hybrid_score",
    "api_score",
    "priority_score"
]:
    if candidate in df.columns:
        score_column = candidate
        break

if score_column is not None:

    df["recommendation_score"] = pd.to_numeric(
        df[score_column],
        errors="coerce"
    ).fillna(0.0)

    print(
        f"✅ Using existing score: {score_column}"
    )

else:

    df["rating_norm"] = minmax(df["rating"])

    df["popularity_norm"] = minmax(
        np.log1p(
            pd.to_numeric(
                df["reviews"],
                errors="coerce"
            ).clip(lower=0)
        )
    )

    df["recommendation_score"] = (
        0.70 * df["rating_norm"]
        + 0.30 * df["popularity_norm"]
    )

    print(
        "⚠️ No recommendation score found."
    )
    print(
        "✅ Fallback rating + popularity score created."
    )


⚠️ No recommendation score found.
✅ Fallback rating + popularity score created.


## 4. Create required planning features

Notebook 15 contains the recommendation features, but the current saved CSV may not contain itinerary-specific columns.

We create them here so this notebook is self-contained.


In [30]:
for column in [
    "name",
    "category",
    "city"
]:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

df["feature_text"] = (
    df["name"].str.lower()
    + " "
    + df["category"].str.lower()
)


def has_any(text, keywords):
    return int(
        any(
            keyword in text
            for keyword in keywords
        )
    )


df["nature"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "beach", "waterfall", "falls",
            "lake", "river", "forest",
            "park", "garden", "valley",
            "mountain", "hill", "nature"
        ]
    )
)

df["history"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "fort", "palace", "museum",
            "heritage", "historical",
            "monument", "haveli", "castle",
            "temple", "church", "mosque"
        ]
    )
)

df["culture"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "temple", "museum", "palace",
            "market", "bazaar", "heritage",
            "fort", "church", "mosque"
        ]
    )
)

df["adventure"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "trek", "rafting", "adventure",
            "ski", "snow", "camp",
            "water sports", "sports"
        ]
    )
)

df["photography"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "viewpoint", "view point",
            "waterfall", "falls", "beach",
            "lake", "fort", "palace",
            "sunset", "garden", "scenic"
        ]
    )
)

df["shopping"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        ["market", "bazaar", "shopping", "mall"]
    )
)

df["religious"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "temple", "church", "mosque",
            "gurudwara", "monastery"
        ]
    )
)

df["family"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "park", "garden", "zoo",
            "museum", "beach", "aquarium",
            "family"
        ]
    )
)


feature_columns = [
    "nature",
    "history",
    "culture",
    "adventure",
    "photography",
    "shopping",
    "religious",
    "family"
]

df["travel_tags"] = df.apply(
    lambda row: ", ".join(
        feature
        for feature in feature_columns
        if row[feature] == 1
    ),
    axis=1
)


## 5. Infer activity type and visit duration

In [31]:
def infer_activity_type(text):
    if "waterfall" in text or "falls" in text:
        return "waterfall"

    if "rafting" in text:
        return "rafting"

    if "trek" in text:
        return "trekking"

    if (
        "viewpoint" in text
        or "view point" in text
    ):
        return "viewpoint"

    if (
        "temple" in text
        or "church" in text
        or "mosque" in text
        or "gurudwara" in text
    ):
        return "religious_site"

    if (
        "fort" in text
        or "palace" in text
        or "museum" in text
        or "heritage" in text
        or "castle" in text
    ):
        return "heritage"

    if (
        "market" in text
        or "bazaar" in text
        or "mall" in text
        or "shopping" in text
    ):
        return "shopping"

    if (
        "beach" in text
        or "lake" in text
        or "river" in text
    ):
        return "nature"

    if (
        "park" in text
        or "forest" in text
        or "garden" in text
    ):
        return "nature"

    if "snow" in text or "ski" in text:
        return "winter_experience"

    return "sightseeing"


df["activity_type"] = df["feature_text"].apply(
    infer_activity_type
)


def estimate_visit_minutes(activity_type):
    duration_map = {
        "waterfall": 90,
        "rafting": 120,
        "trekking": 150,
        "viewpoint": 45,
        "religious_site": 60,
        "heritage": 120,
        "shopping": 90,
        "nature": 90,
        "winter_experience": 90,
        "sightseeing": 60
    }

    return duration_map.get(
        activity_type,
        60
    )


df["estimated_visit_minutes"] = (
    df["activity_type"]
    .apply(estimate_visit_minutes)
)


def estimate_price_level(row):
    text = row["feature_text"]

    if "rafting" in text:
        return 3

    if (
        "ski" in text
        or "adventure" in text
    ):
        return 3

    if (
        "shopping" in text
        or "market" in text
        or "bazaar" in text
        or "mall" in text
    ):
        return 2

    return 1


df["estimated_price_level"] = df.apply(
    estimate_price_level,
    axis=1
)

print("✅ Planning features ready.")


✅ Planning features ready.


## 6. Validate required columns

In [32]:
required_columns = [
    "city",
    "name",
    "latitude",
    "longitude",
    "rating",
    "reviews",
    "recommendation_score",
    "activity_type",
    "estimated_visit_minutes",
    "estimated_price_level"
]

missing = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("✅ All required columns are available.")


✅ All required columns are available.


## 7. Haversine distance

In [33]:
def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):
    R = 6371.0

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return R * c


## 8. Planning configuration

In [34]:
AVERAGE_SPEED_KMPH = 25
DAY_START_MINUTES = 9 * 60
MAX_DAY_MINUTES = 7 * 60

print(
    f"Daily budget: {MAX_DAY_MINUTES} minutes"
)


Daily budget: 420 minutes


## 9. Select city candidates

In [35]:
def select_city_candidates(
    city,
    candidate_count=12
):
    city_clean = city.strip().lower()

    city_df = df[
        df["city"].str.lower() == city_clean
    ].copy()

    if city_df.empty:
        raise ValueError(
            f"City '{city}' not found. "
            f"Available cities: "
            f"{sorted(df['city'].unique())}"
        )

    city_df = (
        city_df
        .sort_values(
            "recommendation_score",
            ascending=False
        )
        .head(
            min(
                candidate_count,
                len(city_df)
            )
        )
        .reset_index(drop=True)
    )

    return city_df


## 10. Build local travel-time matrix

In [36]:
def build_travel_time_matrix(
    city_df
):
    n = len(city_df)

    matrix = np.zeros(
        (n, n),
        dtype=float
    )

    for i in range(n):
        for j in range(n):

            distance = haversine_km(
                city_df.loc[i, "latitude"],
                city_df.loc[i, "longitude"],
                city_df.loc[j, "latitude"],
                city_df.loc[j, "longitude"]
            )

            matrix[i, j] = (
                distance
                / AVERAGE_SPEED_KMPH
                * 60
            )

    return matrix


## 11. Time formatter

In [37]:
def format_time(minutes):
    minutes = int(
        round(minutes)
    )

    hour = (minutes // 60) % 24
    minute = minutes % 60

    suffix = (
        "AM"
        if hour < 12
        else "PM"
    )

    display_hour = hour % 12

    if display_hour == 0:
        display_hour = 12

    return (
        f"{display_hour}:"
        f"{minute:02d} "
        f"{suffix}"
    )


## 12. Generate itinerary with a HARD daily budget

This is the critical fix.

Before a stop is added, the algorithm calculates:

```text
new_used_time =
current_used_time
+ travel_time
+ visit_time
```

A stop is added **only if**:

```text
new_used_time <= MAX_DAY_MINUTES
```

After generation, each day is independently validated.

A post-generation safety check removes the final stop if a floating-point/data issue ever causes a tiny overrun.


In [38]:
def generate_city_itinerary(
    city,
    days=3,
    candidate_count=12,
    max_day_minutes=MAX_DAY_MINUTES
):
    candidates = select_city_candidates(
        city,
        candidate_count
    ).reset_index(drop=True)

    if candidates.empty:
        return pd.DataFrame()

    travel_matrix = build_travel_time_matrix(
        candidates
    )

    remaining = set(
        range(len(candidates))
    )

    all_rows = []

    for day in range(1, days + 1):

        if not remaining:
            break

        # Strongest remaining place starts the day.
        anchor = max(
            remaining,
            key=lambda idx:
                float(
                    candidates.loc[
                        idx,
                        "recommendation_score"
                    ]
                )
        )

        current = anchor
        used_minutes = 0.0
        stop_number = 1

        while remaining:

            feasible = []

            for idx in remaining:

                travel = (
                    0.0
                    if used_minutes == 0
                    else float(
                        travel_matrix[
                            current,
                            idx
                        ]
                    )
                )

                visit = float(
                    candidates.loc[
                        idx,
                        "estimated_visit_minutes"
                    ]
                )

                new_used_minutes = (
                    used_minutes
                    + travel
                    + visit
                )

                # HARD budget condition
                if (
                    new_used_minutes
                    <= max_day_minutes + 1e-9
                ):

                    score = float(
                        candidates.loc[
                            idx,
                            "recommendation_score"
                        ]
                    )

                    efficiency = (
                        score
                        / (1.0 + travel)
                    )

                    feasible.append({
                        "idx": idx,
                        "travel": travel,
                        "visit": visit,
                        "new_used":
                            new_used_minutes,
                        "efficiency":
                            efficiency
                    })

            # Nothing else fits today.
            if not feasible:
                break

            best = max(
                feasible,
                key=lambda item:
                    item["efficiency"]
            )

            idx = best["idx"]
            travel = best["travel"]
            visit = best["visit"]

            # Final guard before writing the row.
            if (
                used_minutes
                + travel
                + visit
                > max_day_minutes + 1e-9
            ):
                break

            arrival = (
                DAY_START_MINUTES
                + used_minutes
                + travel
            )

            departure = (
                arrival + visit
            )

            all_rows.append({
                "city": city,
                "day": day,
                "stop": stop_number,
                "place": candidates.loc[
                    idx,
                    "name"
                ],
                "activity_type": candidates.loc[
                    idx,
                    "activity_type"
                ],
                "arrival": format_time(
                    arrival
                ),
                "departure": format_time(
                    departure
                ),
                "travel_before_minutes":
                    round(travel, 2),
                "visit_minutes":
                    int(round(visit)),
                "estimated_price_level":
                    int(round(
                        candidates.loc[
                            idx,
                            "estimated_price_level"
                        ]
                    )),
                "recommendation_score":
                    round(
                        float(
                            candidates.loc[
                                idx,
                                "recommendation_score"
                            ]
                        ),
                        4
                    ),
                "latitude":
                    float(
                        candidates.loc[
                            idx,
                            "latitude"
                        ]
                    ),
                "longitude":
                    float(
                        candidates.loc[
                            idx,
                            "longitude"
                        ]
                    )
            })

            used_minutes += (
                travel + visit
            )

            current = idx
            remaining.remove(idx)
            stop_number += 1

    result = pd.DataFrame(all_rows)

    if result.empty:
        return result

    # -------------------------------------------------
    # HARD POST-GENERATION VALIDATION / REPAIR
    # -------------------------------------------------

    repaired_rows = []

    for day in sorted(result["day"].unique()):

        day_rows = (
            result[
                result["day"] == day
            ]
            .sort_values("stop")
            .copy()
        )

        while not day_rows.empty:

            total = (
                day_rows["travel_before_minutes"]
                + day_rows["visit_minutes"]
            ).sum()

            if total <= max_day_minutes + 1e-9:
                break

            # Remove the last stop until the budget is safe.
            day_rows = day_rows.iloc[:-1].copy()

        # Re-number remaining stops.
        if not day_rows.empty:
            day_rows["stop"] = range(
                1,
                len(day_rows) + 1
            )
            repaired_rows.append(day_rows)

    if not repaired_rows:
        return pd.DataFrame(
            columns=result.columns
        )

    final_result = pd.concat(
        repaired_rows,
        ignore_index=True
    )

    return final_result


## 13. Test Goa 🌴

In [39]:
goa_itinerary = generate_city_itinerary(
    "Goa",
    days=3,
    candidate_count=12
)

goa_itinerary


,city,day,stop,place,activity_type,arrival,departure,travel_before_minutes,visit_minutes,estimated_price_level,recommendation_score,latitude,longitude
0,Goa,1,1,"Shri Nageshi Temple,",religious_site,9:00 AM,10:00 AM,0.00,60,1,0.8174,15.407464,73.983705
1,Goa,1,2,Chhatrapati Shivaji Maharaj Fort,heritage,10:02 AM,12:02 PM,1.84,120,1,0.5829,15.412500,73.988611
2,Goa,1,3,Sunset View Point Colva,viewpoint,12:43 PM,1:28 PM,41.50,45,1,0.6044,15.274854,73.913585
3,Goa,1,4,Salaulim Dam,sightseeing,2:39 PM,3:39 PM,70.29,60,1,0.6673,15.212704,74.178868
4,Goa,2,1,Dudhsagar Falls,waterfall,9:00 AM,10:30 AM,0.00,90,1,0.7851,15.314438,74.314307
5,Goa,2,2,Bhagwan Mahavir Wildlife Sanctuary,sightseeing,10:38 AM,11:38 AM,8.46,60,1,0.6059,15.333901,74.288354
6,Goa,2,3,Dudhsagar Trek,trekking,11:50 AM,2:20 PM,11.57,150,1,0.5985,15.335836,74.243426
7,Goa,3,1,Keri Beach,nature,9:00 AM,10:30 AM,0.00,90,1,0.7261,15.708774,73.692984
8,Goa,3,2,"Calangute Beach, Goa",nature,11:17 AM,12:47 PM,46.56,90,1,0.6607,15.544721,73.754669
9,Goa,3,3,Sinquerim Fort,heritage,12:59 PM,2:59 PM,12.71,120,1,0.7163,15.498466,73.766404


In [40]:
assert not goa_itinerary.empty
assert goa_itinerary["city"].eq("Goa").all()

print(
    "✅ Goa itinerary is city-specific."
)


✅ Goa itinerary is city-specific.


## 14. Test Jaipur 🏰

In [41]:
jaipur_itinerary = generate_city_itinerary(
    "Jaipur",
    days=3,
    candidate_count=12
)

jaipur_itinerary


,city,day,stop,place,activity_type,arrival,departure,travel_before_minutes,visit_minutes,estimated_price_level,recommendation_score,latitude,longitude
0,Jaipur,1,1,Amber Palace,heritage,9:00 AM,11:00 AM,0.00,120,1,0.8385,26.985487,75.851345
1,Jaipur,1,2,"Sheesh Mahal, Amber Fort",heritage,11:00 AM,1:00 PM,0.18,120,1,0.7330,26.985709,75.850643
2,Jaipur,1,3,Jaigarh Fort,heritage,1:02 PM,3:02 PM,1.77,120,1,0.7310,26.981605,75.844792
3,Jaipur,2,1,Amar Jawan Jyoti,sightseeing,9:00 AM,10:00 AM,0.00,60,1,0.8066,26.895603,75.799957
4,Jaipur,2,2,Central Park,nature,10:03 AM,11:33 AM,3.24,90,1,0.7769,26.904904,75.808722
5,Jaipur,2,3,Albert Hall Museum,heritage,11:36 AM,1:36 PM,3.11,120,1,0.7572,26.911585,75.819412
6,Jaipur,2,4,Hawa Mahal,sightseeing,1:40 PM,2:40 PM,3.75,60,1,0.7835,26.924046,75.826714
7,Jaipur,2,5,Jantar Mantar,sightseeing,2:41 PM,3:41 PM,0.55,60,1,0.7404,26.924762,75.824560
8,Jaipur,3,1,Nahargarh Fort,heritage,9:00 AM,11:00 AM,0.00,120,1,0.7596,26.940166,75.817037
9,Jaipur,3,2,Gaitor Ki Chhatriyan,sightseeing,11:02 AM,12:02 PM,1.97,60,1,0.7327,26.943069,75.824653


In [42]:
assert not jaipur_itinerary.empty
assert jaipur_itinerary["city"].eq("Jaipur").all()

print(
    "✅ Jaipur itinerary is city-specific."
)


✅ Jaipur itinerary is city-specific.


## 15. Test all supported cities

In [43]:
supported_cities = sorted(
    df["city"].unique()
)

city_itineraries = {}

for city in supported_cities:

    plan = generate_city_itinerary(
        city,
        days=3,
        candidate_count=12
    )

    city_itineraries[city] = plan

    print(
        f"{city:12} → "
        f"{len(plan):2} stops"
    )


Goa          → 10 stops
Jaipur       → 11 stops
Manali       → 12 stops
Rishikesh    → 12 stops
Shimla       → 12 stops
Udaipur      → 12 stops


## 16. Day distribution

In [44]:
distribution_rows = []

for city, plan in city_itineraries.items():

    counts = (
        plan["day"]
        .value_counts()
        .to_dict()
        if not plan.empty
        else {}
    )

    for day in range(1, 4):

        distribution_rows.append({
            "city": city,
            "day": day,
            "stops": counts.get(
                day,
                0
            )
        })

distribution_df = pd.DataFrame(
    distribution_rows
)

distribution_df


,city,day,stops
0,Goa,1,4
1,Goa,2,3
2,Goa,3,3
3,Jaipur,1,3
4,Jaipur,2,5
5,Jaipur,3,3
6,Manali,1,4
7,Manali,2,5
8,Manali,3,3
9,Rishikesh,1,6


## 17. Exact daily time check

In [45]:
all_plans = pd.concat(
    [
        plan
        for plan in city_itineraries.values()
        if not plan.empty
    ],
    ignore_index=True
)

daily_time = (
    all_plans
    .assign(
        total_minutes=lambda x:
            x["travel_before_minutes"]
            + x["visit_minutes"]
    )
    .groupby(
        ["city", "day"]
    )["total_minutes"]
    .sum()
    .reset_index()
)

daily_time["within_budget"] = (
    daily_time["total_minutes"]
    <= MAX_DAY_MINUTES + 1e-9
)

daily_time


,city,day,total_minutes,within_budget
0,Goa,1,398.63,True
1,Goa,2,320.03,True
2,Goa,3,359.27,True
3,Jaipur,1,361.95,True
4,Jaipur,2,400.65,True
5,Jaipur,3,305.25,True
6,Manali,1,394.26,True
7,Manali,2,393.59,True
8,Manali,3,315.14,True
9,Rishikesh,1,352.81,True


## 18. HARD assertion

This must be completely green.

If this assertion fails, the itinerary generator should be considered invalid and we should fix the planner rather than ignoring the result.


In [46]:
assert daily_time[
    "within_budget"
].all()

print(
    "✅ Every generated city/day is within "
    f"the {MAX_DAY_MINUTES}-minute budget."
)


✅ Every generated city/day is within the 420-minute budget.


## 19. Verify no itinerary mixes cities

In [47]:
city_integrity = (
    all_plans
    .groupby("city")["city"]
    .nunique()
    .reset_index(
        name="unique_city_values"
    )
)

city_integrity["valid"] = (
    city_integrity[
        "unique_city_values"
    ] == 1
)

city_integrity


,city,unique_city_values,valid
0,Goa,1,True
1,Jaipur,1,True
2,Manali,1,True
3,Rishikesh,1,True
4,Shimla,1,True
5,Udaipur,1,True


In [48]:
assert city_integrity["valid"].all()

print(
    "✅ No itinerary mixes destinations."
)


✅ No itinerary mixes destinations.


## 20. Itinerary statistics

In [49]:
itinerary_statistics = (
    all_plans
    .groupby("city")
    .agg(
        scheduled_places=("place", "count"),
        total_visit_minutes=(
            "visit_minutes",
            "sum"
        ),
        estimated_travel_minutes=(
            "travel_before_minutes",
            "sum"
        ),
        average_recommendation_score=(
            "recommendation_score",
            "mean"
        ),
        total_price_units=(
            "estimated_price_level",
            "sum"
        )
    )
    .reset_index()
)

itinerary_statistics


,city,scheduled_places,total_visit_minutes,estimated_travel_minutes,average_recommendation_score,total_price_units
0,Goa,10,885,192.93,0.676460,10
1,Jaipur,11,1050,17.85,0.759136,11
2,Manali,12,1035,67.99,0.676858,13
3,Rishikesh,12,855,40.59,0.699908,14
4,Shimla,12,765,22.46,0.674667,12
5,Udaipur,12,1020,27.26,0.762925,12


## 21. Save unified itinerary

In [50]:
output_dir = Path(
    "../data/processed"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

unified_path = (
    output_dir
    / "multi_city_itineraries.csv"
)

all_plans.to_csv(
    unified_path,
    index=False
)

print(
    f"✅ Saved: {unified_path}"
)


✅ Saved: ..\data\processed\multi_city_itineraries.csv


## 22. Save one itinerary per city

In [51]:
city_dir = (
    output_dir
    / "city_itineraries"
)

city_dir.mkdir(
    parents=True,
    exist_ok=True
)

for city, plan in city_itineraries.items():

    filename = (
        city.lower()
        .replace(" ", "_")
        + "_itinerary.csv"
    )

    plan.to_csv(
        city_dir / filename,
        index=False
    )

print(
    f"✅ Per-city itineraries saved in: {city_dir}"
)


✅ Per-city itineraries saved in: ..\data\processed\city_itineraries
